<a href="https://colab.research.google.com/github/Erickzlyn/FUNDAI-Laboratories-Redondo/blob/main/Lab4_Logic_KR_Redondo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Logic and Knowledge Representation

## Fundamentals of Artificial Intelligence

**Name:** Erick Lyndon S. Redondo
**Course:** CS-FUNDAI
**Section:** 09282-FUNDAI
**Date:**  9/17/2026

**GitHub URL:** https://github.com/Erickzlyn/FUNDAI-Laboratories-Redondo.git

## Description
This laboratory uses Python and SymPy to perform truth table generation, satisfiability checking, theorem proving, and logical deduction.

In [1]:
from sympy import symbols, And, Or, Not, Implies, Equivalent, satisfiable
from itertools import product

In [2]:
P, Q, R = symbols('P Q R')

In [3]:
def print_truth_table(expression, symbol_list):
    header = [str(s) for s in symbol_list] + [str(expression)]
    print(" | ".join(header))
    print("-" * (5 * len(header)))

    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        result = bool(expression.subs(mapping))
        row = [str(v) for v in values] + [str(result)]
        print(" | ".join(row))

    print()


print("Negation: NOT P")
print_truth_table(Not(P), [P])

print("Conjunction: P AND Q")
print_truth_table(And(P, Q), [P, Q])

print("Disjunction: P OR Q")
print_truth_table(Or(P, Q), [P, Q])

print("Implication: P -> Q")
print_truth_table(Implies(P, Q), [P, Q])

print("Biconditional: P <-> Q")
print_truth_table(Equivalent(P, Q), [P, Q])

Negation: NOT P
P | ~P
----------
False | True
True | False

Conjunction: P AND Q
P | Q | P & Q
---------------
False | False | False
False | True | False
True | False | False
True | True | True

Disjunction: P OR Q
P | Q | P | Q
---------------
False | False | False
False | True | True
True | False | True
True | True | True

Implication: P -> Q
P | Q | Implies(P, Q)
---------------
False | False | True
False | True | True
True | False | False
True | True | True

Biconditional: P <-> Q
P | Q | Equivalent(P, Q)
---------------
False | False | True
False | True | False
True | False | False
True | True | True



In [4]:
def is_tautology(expression, symbol_list):
    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        if not bool(expression.subs(mapping)):
            return False
    return True


law_of_excluded_middle = Or(P, Not(P))
contradiction = And(P, Not(P))
simple_implication = Implies(P, Q)

print("P OR NOT P is a tautology:", is_tautology(law_of_excluded_middle, [P]))
print("P AND NOT P is a tautology:", is_tautology(contradiction, [P]))
print("P -> Q is a tautology:", is_tautology(simple_implication, [P, Q]))

P OR NOT P is a tautology: True
P AND NOT P is a tautology: False
P -> Q is a tautology: False


In [5]:
def is_satisfiable(expression):
    return satisfiable(expression) is not False


print("P AND NOT P is satisfiable:", is_satisfiable(And(P, Not(P))))
print("P OR Q is satisfiable:", is_satisfiable(Or(P, Q)))
print("P -> Q is satisfiable:", is_satisfiable(Implies(P, Q)))

P AND NOT P is satisfiable: False
P OR Q is satisfiable: True
P -> Q is satisfiable: True


In [6]:
def to_conjunction(kb):
    if isinstance(kb, list):
        return And(*kb)
    return kb


def kb_entails(kb, conclusion):
    kb_expression = to_conjunction(kb)
    counter_check = And(kb_expression, Not(conclusion))
    return satisfiable(counter_check) is False


def check_entailment(kb, conclusion, label="Query"):
    holds = kb_entails(kb, conclusion)
    kb_expression = to_conjunction(kb)
    counterexample = satisfiable(And(kb_expression, Not(conclusion)))

    print(label)

    if holds:
        print("Result: Entailment holds.")
    else:
        print("Result: Entailment does not hold.")
        print("Counterexample model:", counterexample)

    print("-" * 60)
    return holds

In [7]:
Rain, Wet = symbols('Rain Wet')

kb_rain = [
    Implies(Rain, Wet),
    Rain
]

check_entailment(kb_rain, Wet, "Theorem Proving: Rain example")

Theorem Proving: Rain example
Result: Entailment holds.
------------------------------------------------------------


True

In [8]:
kb_invalid = [
    Implies(Rain, Wet),
    Wet
]

check_entailment(kb_invalid, Rain, "Invalid Inference: Affirming the consequent")

Invalid Inference: Affirming the consequent
Result: Entailment does not hold.
Counterexample model: {Wet: True, Rain: False}
------------------------------------------------------------


False

In [9]:
print("Logical Deduction Rules")
print("=" * 60)

# Modus Ponens
check_entailment(
    [P, Implies(P, Q)],
    Q,
    "Modus Ponens: P, P -> Q, therefore Q"
)

# Modus Tollens
check_entailment(
    [Not(Q), Implies(P, Q)],
    Not(P),
    "Modus Tollens: NOT Q, P -> Q, therefore NOT P"
)

Logical Deduction Rules
Modus Ponens: P, P -> Q, therefore Q
Result: Entailment holds.
------------------------------------------------------------
Modus Tollens: NOT Q, P -> Q, therefore NOT P
Result: Entailment holds.
------------------------------------------------------------


True

## Grounded First-Order Logic Example

Full First-Order Logic includes objects and quantifiers.
For this laboratory, we demonstrate a simple grounded FOL example
by converting FOL atoms into propositional symbols.

English:
- All humans are mortal.
- Socrates is human.
- Therefore, Socrates is mortal.

Grounded propositional form:
- Human_Socrates -> Mortal_Socrates

In [10]:
Human_Socrates, Mortal_Socrates = symbols('Human_Socrates Mortal_Socrates')

kb_socrates = [
    Implies(Human_Socrates, Mortal_Socrates),
    Human_Socrates
]

check_entailment(
    kb_socrates,
    Mortal_Socrates,
    "Grounded FOL: Socrates is mortal"
)

Grounded FOL: Socrates is mortal
Result: Entailment holds.
------------------------------------------------------------


True

In [11]:
def make_human_mortal_kb(constants):
    kb = []
    human = {}
    mortal = {}

    for name in constants:
        h, m = symbols(f'Human_{name} Mortal_{name}')
        human[name] = h
        mortal[name] = m
        kb.append(Implies(h, m))

    return kb, human, mortal

In [12]:
constants = ["Socrates", "Plato"]

kb_people, human, mortal = make_human_mortal_kb(constants)

# Add facts
kb_people.append(human["Socrates"])
kb_people.append(human["Plato"])

# Query: Is Plato mortal?
check_entailment(
    kb_people,
    mortal["Plato"],
    "Grounded FOL with multiple constants: Is Plato mortal?"
)

Grounded FOL with multiple constants: Is Plato mortal?
Result: Entailment holds.
------------------------------------------------------------


True

## Guide Questions and Answers

### 1. What is the difference between syntax and semantics?
**Answer:** Syntax is just the grammar rules — how symbols and logical operators are allowed to be put together. Semantics is about what those combinations actually mean, like whether a statement ends up true or false.

### 2. Why is `P -> Q` true when `P` is false?
**Answer:** An implication only breaks its promise if `P` is true but `Q` turns out false. If `P` is never true in the first place, there's nothing to break, so we just count it as true by default.

### 3. What does it mean for a knowledge base to entail a conclusion?
**Answer:** It means that whenever everything in the knowledge base is true, the conclusion has to be true too — there's no possible situation where the KB holds but the conclusion doesn't.

### 4. How does theorem proving use satisfiability checking?
**Answer:** To prove something follows from the KB, we try to find a case where the KB is true but the conclusion is false. If no such case exists (it's unsatisfiable), that proves the conclusion must be true.

### 5. What is one limitation of propositional logic compared to First-Order Logic?
**Answer:** Propositional logic can't talk about objects or use quantifiers like "all" or "some." It can only handle fixed true/false statements, while First-Order Logic can express general rules that apply across many objects at once.

## Reflection

### Challenges Encountered
- Getting the truth table function to line up correctly took a bit of trial and error, especially formatting the output so it was readable.
- Understanding why unsatisfiability of the negated conclusion proves entailment took some time to really click.

### What I Learned
- How SymPy can be used to represent and evaluate logical expressions instead of doing truth tables by hand.
- The connection between satisfiability checking and proving whether one statement logically follows from others.